# Phase 4 — Agent A3 : Sentiment Analyser
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A3 (`agents/a3_sentiment.py`) :
- Classification du sentiment global du corpus (positive / neutral / negative)
- Production du `Score_Sentiment` [0-100]
- Comportement de fallback quand le LLM est indisponible

> Les appels LLM sont **mockés** dans ce notebook — aucun GPU requis.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a3_sentiment import a3_sentiment

## 1. Corpus de test

In [ ]:
CLEANED_COMMENTS = [
    {"text": "This is a great tutorial on machine learning!", "cleaned_text": "this is a great tutorial on machine learning!"},
    {"text": "Very informative, learned a lot about neural networks.", "cleaned_text": "very informative, learned a lot about neural networks."},
    {"text": "Excellent explanation of gradient descent.", "cleaned_text": "excellent explanation of gradient descent."},
    {"text": "Best course I have ever taken.", "cleaned_text": "best course i have ever taken."},
    {"text": "Could you cover backpropagation in more detail?", "cleaned_text": "could you cover backpropagation in more detail?"},
]

STATE = {"cleaned_comments": CLEANED_COMMENTS}
print(f"{len(CLEANED_COMMENTS)} commentaires chargés.")

## 2. A3 avec LLM mocké — sentiment positif

In [ ]:
mock_response = MagicMock()
mock_response.content = json.dumps({
    "sentiment_label": "positive",
    "sentiment_score": 0.82,
    "rationale": "The corpus expresses strong enthusiasm and positive feedback.",
})

mock_llm = MagicMock()
mock_llm.invoke.return_value = mock_response

with patch("agents.a3_sentiment.get_llm", return_value=mock_llm):
    result = a3_sentiment(STATE)

s = result["sentiment"]
print(f"Label    : {s['sentiment_label']}")
print(f"Score    : {s['sentiment_score']} / 100")
print(f"Rationale: {s['rationale']}")

## 3. Fallback — LLM indisponible

In [ ]:
with patch("agents.a3_sentiment.get_llm", return_value=None):
    fallback = a3_sentiment(STATE)

s = fallback["sentiment"]
print(f"Fallback label : {s['sentiment_label']}  (attendu: neutral)")
print(f"Fallback score : {s['sentiment_score']}    (attendu: 50.0)")
assert s["sentiment_label"] == "neutral"
assert s["sentiment_score"] == 50.0
print("Fallback conforme.")

## Résumé A3

| Comportement | Résultat |
|---|---|
| Sentiment positif (score 0.82) | Score_Sentiment = 82.0 / 100 |
| Score [0,1] → [0,100] | Multiplication par 100 confirmée |
| LLM indisponible | Fallback neutral / 50.0 |
| Erreur JSON | Fallback neutral / 50.0 |

> **Modèle cible Kaggle** : Phi-3-mini-4k-instruct (float16, ~4 Go VRAM)